In [1]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets

Visualize nnUNet predicted probability maps overlayed on CT slices:

In [ ]:
# ============================================================
# Utility
# ============================================================

def load_nnUNet_probs(npz_path):
    data = np.load(npz_path)
    probs = data["probabilities"]  # (C, X, Y, Z)
    fg_probs = probs[1]    
    print(np.where(np.any(fg_probs > 0.2, axis=(1, 2)))[0])        # second channel = foreground
    return fg_probs

def load_ct_volume(ct_path):
    ct = nib.load(str(ct_path)).get_fdata()
    ct = np.transpose(ct, (2, 1, 0))  # match probabilities orientation
    return ct

def window_ct_hu(ct, level=50, width=350):
    lower = level - width / 2.0
    upper = level + width / 2.0
    ct = np.clip(ct, lower, upper)
    ct = (ct - lower) / (upper - lower + 1e-6)
    return ct

# ============================================================
# Interactive Visualization
# ============================================================

def visualize_prob_overlay_slider(ct_volume, prob_volume, axis="axial", cmap_name="jet"):
    axis_map = {"axial":0, "coronal":1, "sagittal":2}
    ax = axis_map[axis]
    n_slices = ct_volume.shape[ax]

    def plot_slice(idx):
        if axis == "axial":
            ct_slice = ct_volume[idx, :, :]
            prob_slice = prob_volume[idx, :, :]
        elif axis == "coronal":
            ct_slice = ct_volume[:, idx, :]
            prob_slice = prob_volume[:, idx, :]
        elif axis == "sagittal":
            ct_slice = ct_volume[:, :, idx]
            prob_slice = prob_volume[:, :, idx]

        ct_img = window_ct_hu(ct_slice)

        plt.figure(figsize=(6,6))
        plt.imshow(ct_img, cmap="gray", origin="lower")
        plt.imshow(prob_slice, cmap=cmap_name, alpha=0.2, origin="lower", vmin=0, vmax=1)
        plt.colorbar(label="Foreground probability")
        plt.title(f"{axis.capitalize()} slice {idx}/{n_slices-1}")
        plt.axis("off")
        plt.show()

    slice_slider = widgets.IntSlider(
        value=n_slices//2, min=0, max=n_slices-1, step=1,
        description=f'{axis.capitalize()} slice', continuous_update=False
    )

    widgets.interact(plot_slice, idx=slice_slider)

# ============================================================
# Example usage
# ============================================================

if __name__ == "__main__":
    
    uid = 339
    ct_path = f"/data/colon_cancer/CC_Detection/raw_data/Dataset107_CC/imagesTs/{uid}_0000.nii.gz"
    npz_path = f"/data/colon_cancer/CC_Detection/raw_data/Dataset107_CC/predictionsTs_cascade/{uid}.npz"

    ct = load_ct_volume(ct_path)
    fg_probs = load_nnUNet_probs(npz_path)

    visualize_prob_overlay_slider(ct, fg_probs, axis="axial")


Visualize Data:

In [3]:
import os
import blosc2
import nibabel as nib
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
import ipywidgets as widgets
import scipy.ndimage as ndi
from pathlib import Path
from matplotlib.colors import ListedColormap
from scipy.ndimage import distance_transform_edt as dist
from monai.transforms import RemoveSmallObjects, KeepLargestConnectedComponent
from acvl_utils.morphology.morphology_helper import remove_all_but_largest_component

# ============================================================
# Loading & Image Utilities
# ============================================================

def load_volume(path):
    """Load 3D volumes supporting NIfTI, NPZ, NPY, and B2ND formats."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    
    if path.suffix == ".npy":
        return np.load(path)
    elif path.suffix == ".npz":
        data = np.load(path)
        return data[list(data.keys())[0]]
    elif path.suffix in [".nii", ".gz"]:
        return nib.load(str(path)).get_fdata()
    elif path.suffix == ".b2nd":
        schunk = blosc2.open(path, mode="r")
        return schunk[:][0] if len(schunk[:]) == 1 else schunk[:][1]
    raise ValueError(f"Unsupported format: {path.suffix}")

def window_ct_hu(ct_hu, level=50, width=350):
    """Apply soft tissue windowing and normalize to [0, 1]."""
    lower, upper = level - width / 2.0, level + width / 2.0
    return np.clip((ct_hu - lower) / (upper - lower + 1e-6), 0, 1)

def resample_label_to_image(label_path, ref_path):
    """Resample label SITK image to match reference image grid."""
    lbl_sitk = sitk.ReadImage(str(label_path))
    ref_sitk = sitk.ReadImage(str(ref_path))
    resampled = sitk.Resample(
        lbl_sitk, ref_sitk, sitk.Transform(),
        sitk.sitkNearestNeighbor, 0, lbl_sitk.GetPixelID()
    )
    return sitk.GetArrayFromImage(resampled).transpose(2, 1, 0)

# ============================================================
# Morphology & Post-Processing
# ============================================================

def interpolate_missing_slices(label, axis="axial"):
    """Fill gaps in segmentation using linear interpolation in DT space."""
    lbl = label.copy()
    # Transpose to make interpolation axis the first dimension
    axes_map = {"axial": (0, 1, 2), "coronal": (1, 0, 2), "sagittal": (2, 1, 0)}
    lbl = np.transpose(lbl, axes_map[axis])
    
    annotated = np.where(np.any(lbl > 0, axis=(1, 2)))[0]
    if len(annotated) < 2: return label

    # Signed distance transform
    dt = np.array([dist(~(slice > 0)) - dist(slice > 0) for slice in lbl])
    
    for k in range(lbl.shape[0]):
        if k not in annotated:
            prev_a = annotated[annotated < k]
            next_a = annotated[annotated > k]
            if len(prev_a) > 0 and len(next_a) > 0:
                kp, kn = prev_a[-1], next_a[0]
                w = (k - kp) / (kn - kp)
                lbl[k] = ((1 - w) * dt[kp] + w * dt[kn]) < 0

    # Reverse transpose
    reverse_map = {"axial": (0, 1, 2), "coronal": (1, 0, 2), "sagittal": (2, 1, 0)}
    return np.transpose(lbl, reverse_map[axis]).astype(label.dtype)

# ============================================================
# Visualization Overlays
# ============================================================

def get_overlay_config(label_volume, mode="labels"):
    """Returns (overlay_data, cmap, vminmax) based on mode."""
    if mode == "large_component":
        mask_keep = remove_all_but_largest_component(label_volume)
        overlay = np.zeros_like(label_volume, dtype=np.int32)
        overlay[(label_volume > 0) & (~mask_keep)] = 1  # Removed (Red)
        overlay[mask_keep] = 2                         # Kept (Green)
        cmap = ListedColormap([[0,0,0,0], [1,0,0,0.35], [0,1,0,0.35]])
        return overlay, cmap, (0, 2)
    
    elif mode == "small_objects":
        cleaned = RemoveSmallObjects(min_size=200)(label_volume)
        overlay = np.zeros_like(label_volume, dtype=np.int32)
        overlay[cleaned > 0] = 1                       # Large (Green)
        overlay[(label_volume > 0) & (cleaned == 0)] = 2 # Small (Red)
        cmap = ListedColormap([[0,0,0,0], [0,1,0,0.25], [1,0,0,0.25]])
        return overlay, cmap, (0, 2)
    
    else: # Default multi-class labels
        unique = np.unique(label_volume)
        colors = [[0,0,0,0], [1,0,0,0.35], [0,1,0,0.35], [0,0,1,0.35], [1,1,0,0.35]]
        cmap = ListedColormap(colors[:len(unique)])
        return label_volume.astype(np.int32), cmap, (0, int(unique.max()))

# ============================================================
# Main Interactive Visualization
# ============================================================

def visualize_case(ct_path, label_path, uid, target="N/A", axis="axial", mode="labels", **kwargs):
    """Interactive slice viewer for CT and Segmentation overlays."""
    ct = load_volume(ct_path)
    label = resample_label_to_image(label_path, ct_path) if kwargs.get('resample', False) else load_volume(label_path)

    # Orientation correction for non-blosc formats
    if Path(ct_path).suffix != ".b2nd":
        ct, label = ct.transpose(2, 1, 0), label.transpose(2, 1, 0)

    # --- Print Annotated Slices ---
    reduce_axes = {"axial": (1, 2), "coronal": (0, 2), "sagittal": (0, 1)}[axis]
    annotated_slices = np.where(np.any(label > 0, axis=reduce_axes))[0]
    
    if len(annotated_slices) > 0:
        print(f"✅ UID {uid}: Annotations found on {len(annotated_slices)} slices.")
        print(f"Indices ({axis}): {annotated_slices.tolist()}")
    else:
        print(f"⚠️ UID {uid}: No annotations found in the label volume for {axis} view.")

    overlay_3d, cmap, vminmax = get_overlay_config(label, mode)
    
    def plot_slice(idx):
        # Extract slices based on orientation
        slicer = {"axial": (idx, slice(None), slice(None)), 
                  "coronal": (slice(None), idx, slice(None)), 
                  "sagittal": (slice(None), slice(None), idx)}
        
        img_slice = window_ct_hu(ct[slicer[axis]])
        ovl_slice = overlay_3d[slicer[axis]]

        plt.figure(figsize=(6, 6))
        plt.imshow(img_slice, cmap="gray", origin="lower")
        plt.imshow(ovl_slice, cmap=cmap, origin="lower", vmin=vminmax[0], vmax=vminmax[1])
        plt.title(f"UID: {uid} ({target}) | {axis} idx: {idx}")
        plt.axis("off")
        plt.show()

    n_slices = ct.shape[{"axial":0, "coronal":1, "sagittal":2}[axis]]
    
    # Start slider at the first annotated slice if available, otherwise middle
    start_val = annotated_slices[0] if len(annotated_slices) > 0 else n_slices // 2
    
    widgets.interact(plot_slice, idx=widgets.IntSlider(
        min=0, max=n_slices-1, value=int(start_val), description='Slice'
    ))

def run_batch_viz(uids, img_dir, lbl_dir, split_csv, **kwargs):
    """Crawl directories and visualize a batch of UIDs."""
    df = pd.read_csv(split_csv)
    for uid in uids:
        # Search for files with various naming patterns
        img_file = next(Path(img_dir).glob(f"*{uid}*"), None)
        lbl_file = next(Path(lbl_dir).glob(f"*{uid}*"), None)

        if not (img_file and lbl_file): continue

        row = df[df["UID"] == uid]
        target = "CC" if row["target"].values[0] == 1 else "DIV"
        visualize_case(img_file, lbl_file, uid, target, **kwargs)

# ============================================================
# Usage Example
# ============================================================

if __name__ == "__main__":
    # Example Config
    UID_LIST = [13]
    IMAGES = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/imagesTs"
    LABELS = "/data/colon_cancer/CC_Detection/raw_data/Dataset100_CC/predictionsTs"
    SPLITS = "/data/colon_cancer/Classifier/ColonCancer/splits.csv"

    run_batch_viz(
        uids=UID_LIST, 
        img_dir=IMAGES, 
        lbl_dir=LABELS, 
        split_csv=SPLITS, 
        mode="large_component", 
        axis="axial"
    )

✅ UID 13: Annotations found on 18 slices.
Indices (axial): [28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 77, 78, 79, 80]


interactive(children=(IntSlider(value=28, description='Slice', max=104), Output()), _dom_classes=('widget-inte…